In [0]:
import requests
import pandas as pd

account_name = "mystorageaccountpr"
container_name = "raw-landing"
sas_token = "sp=rwl&st=2026-07-16T16:27:37Z&se=2026-07-19T00:42:37Z&spr=https&sv=2026-02-06&sr=c&sig=1Htrh5n%2BuxjkwdsonzQA9HOPHp%2B5qU25zhK6dgbM6ZA%3D"

payment_types = ["visa", "americanexpress", "mastercard"]
dfs = []

for ptype in payment_types:
    blob_path = f"raw/{ptype}/{ptype}.csv"
    url = f"https://{account_name}.blob.core.windows.net/{container_name}/{blob_path}?{sas_token}"
    resp = requests.get(url)
    print(f"{ptype}: status {resp.status_code}")
    if resp.status_code == 200:
        local_path = f"/tmp/{ptype}.csv"
        with open(local_path, "wb") as f:
            f.write(resp.content)
        df = pd.read_csv(local_path)
        df["_source_file"] = f"{ptype}.csv"
        dfs.append(df)
        print(f"  -> {df.shape[0]} rows loaded")
    else:
        print(f"  -> FAILED: {resp.text[:200]}")

visa: status 200
  -> 4410 rows loaded
americanexpress: status 200
  -> 1416 rows loaded
mastercard: status 200
  -> 1420 rows loaded


In [0]:
bronze_pd = pd.concat(dfs, ignore_index=True)
bronze_pd["_ingested_at"] = pd.Timestamp.now()

bronze_df = spark.createDataFrame(bronze_pd)

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
bronze_df.write.format("delta").mode("overwrite").saveAsTable("bronze.retail_sales_raw")

print(f"Total bronze rows: {bronze_df.count()}")

Total bronze rows: 7246


In [0]:
display(spark.sql("SELECT * FROM bronze.retail_sales_raw LIMIT 10"))

transaction_id,transactional_date,product_id,customer_id,payment,loyalty_card,cost,quantity,price,_source_file,_ingested_at
3624.0,25-02-2022 13:04,P0209,4.0,visa,F,6.02,1.0,7.29,visa.csv,2026-07-17T06:54:49.175Z
null,null,null,null,null,null,null,null,null,visa.csv,2026-07-17T06:54:49.175Z
null,null,null,null,null,null,null,null,null,visa.csv,2026-07-17T06:54:49.175Z
null,null,null,null,null,null,null,null,null,visa.csv,2026-07-17T06:54:49.175Z
null,null,null,null,null,null,null,null,null,visa.csv,2026-07-17T06:54:49.175Z
3629.0,25-02-2022 23:40,P0658,3.0,visa,T,2.92,5.0,4.09,visa.csv,2026-07-17T06:54:49.175Z
null,null,null,null,null,null,null,null,null,visa.csv,2026-07-17T06:54:49.175Z
null,null,null,null,null,null,null,null,null,visa.csv,2026-07-17T06:54:49.175Z
3632.0,26-02-2022 09:13,P0715,4.0,null,F,7.6,3.0,8.29,visa.csv,2026-07-17T06:54:49.175Z
null,null,null,null,null,null,null,null,null,visa.csv,2026-07-17T06:54:49.175Z
